# Ex.2 Eight point algorithm for Essential Matrix estimation

In [ ]:
import cv2
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.cm as cm # Import colormaps
import random
import numpy.linalg as la

### Step 1: Load images and calibration data

Load the calibration matrix from the JSON file

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

def pick_existing(paths):
    for p in paths:
        candidate = PROJECT_ROOT / p
        if candidate.exists():
            return candidate
    raise FileNotFoundError('None of the candidate paths exist: ' + str(paths))

# Mapping requested:
# - foto dritta  -> img_1  (img1)
# - foto storta  -> img_2  (img2)
calib_path = PROJECT_ROOT / 'calibration.json'
img1_path = pick_existing([
    'Photos2/Img_1.jpeg',
    'Photos2/Img_2.jpeg',
    'Photos/img_1.jpeg',
    'Photos/img1.jpeg',
    'Photos2/img_1.jpeg',
    'Photos2/img1.jpeg',
])
img2_path = pick_existing([
    'Photos2/Img2.jpeg',
    'Photos2/Img_2.jpeg',
    'Photos/img_2.jpeg',
    'Photos/img2.jpeg',
    'Photos2/img2.jpeg',
])

with open(calib_path, 'r') as f:
    calib = json.load(f)

K = np.array(calib['camera_matrix'])

img1 = cv2.imread(str(img1_path))
img2 = cv2.imread(str(img2_path))

if img1 is None or img2 is None:
    raise FileNotFoundError('Could not load selected image files')

img1_rgb = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
img2_rgb = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

print(f'Loaded calibration from: {calib_path}')
print(f'Loaded image 1 (dritta) from: {img1_path}')
print(f'Loaded image 2 (storta) from: {img2_path}')
print(f'Image 1 Size: {img1_rgb.shape[1]}x{img1_rgb.shape[0]}')
print(f'Image 2 Size: {img2_rgb.shape[1]}x{img2_rgb.shape[0]}')

### Step 2a: Automatic SIFT correspondences on floor

SIFT è usato per aggiungere corrispondenze nel pavimento/aiuola, dove i punti manuali non bastano.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

# ---------------------------------------------------------------------
# FAST PATH: use saved results to avoid grid-search
# ---------------------------------------------------------------------
USE_SAVED = True
SAVED_PATH = 'sift_inliers_best.npz'

if USE_SAVED and Path(SAVED_PATH).exists():
    data = np.load(SAVED_PATH, allow_pickle=True)
    points1 = data['points1'].astype(np.float32)
    points2 = data['points2'].astype(np.float32)
    F = data['F'].astype(np.float64)

    print('Caricati risultati salvati da:', SAVED_PATH)
    print(f'Inliers caricati: {len(points1)}')

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(img1_rgb)
    axes[0].scatter(points1[:, 0], points1[:, 1], s=8, c='lime')
    axes[0].set_title(f'Image 1 - Inlier points ({len(points1)})')
    axes[0].axis('off')

    axes[1].imshow(img2_rgb)
    axes[1].scatter(points2[:, 0], points2[:, 1], s=8, c='cyan')
    axes[1].set_title(f'Image 2 - Inlier points ({len(points2)})')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

else:
    with open(calib_path, 'r') as f:
        calib = json.load(f)

    K = np.array(calib['camera_matrix'], dtype=np.float64)
    dist = np.array(calib['dist_coeffs'], dtype=np.float64).reshape(-1, 1)

    # Usa le immagini già selezionate nello Step 1
    img1_local = cv2.imread(str(img1_path))
    img2_local = cv2.imread(str(img2_path))
    if img1_local is None or img2_local is None:
        raise FileNotFoundError('Immagini non trovate nei path selezionati nello Step 1')

    # Undistort
    h1, w1 = img1_local.shape[:2]
    h2, w2 = img2_local.shape[:2]
    newK1, _ = cv2.getOptimalNewCameraMatrix(K, dist, (w1, h1), 1, (w1, h1))
    newK2, _ = cv2.getOptimalNewCameraMatrix(K, dist, (w2, h2), 1, (w2, h2))
    u1 = cv2.undistort(img1_local, K, dist, None, newK1)
    u2 = cv2.undistort(img2_local, K, dist, None, newK2)

    # Grayscale + CLAHE
    g1 = cv2.cvtColor(u1, cv2.COLOR_BGR2GRAY)
    g2 = cv2.cvtColor(u2, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    g1 = clahe.apply(g1)
    g2 = clahe.apply(g2)

    # Maschera anti-bordi neri dovuti a undistort
    mask1 = (cv2.cvtColor(u1, cv2.COLOR_BGR2GRAY) > 8).astype(np.uint8) * 255
    mask2 = (cv2.cvtColor(u2, cv2.COLOR_BGR2GRAY) > 8).astype(np.uint8) * 255

    def sampson_errors(F, pts1, pts2):
        pts1_h = np.hstack([pts1, np.ones((pts1.shape[0], 1), dtype=np.float64)])
        pts2_h = np.hstack([pts2, np.ones((pts2.shape[0], 1), dtype=np.float64)])

        Fx1 = (F @ pts1_h.T).T
        Ftx2 = (F.T @ pts2_h.T).T
        x2tFx1 = np.sum(pts2_h * Fx1, axis=1)

        denom = Fx1[:, 0] ** 2 + Fx1[:, 1] ** 2 + Ftx2[:, 0] ** 2 + Ftx2[:, 1] ** 2
        denom = np.maximum(denom, 1e-12)
        return (x2tFx1 ** 2) / denom

    def knn_matches(des1, des2, matcher_type='bf'):
        if matcher_type == 'flann':
            index_params = dict(algorithm=1, trees=8)
            search_params = dict(checks=80)
            matcher = cv2.FlannBasedMatcher(index_params, search_params)
        else:
            matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
        return matcher.knnMatch(des1, des2, k=2)

    def unique_matches(good_matches):
        good_sorted = sorted(good_matches, key=lambda m: m.distance)
        used_q = set()
        used_t = set()
        unique = []
        for m in good_sorted:
            if m.queryIdx in used_q or m.trainIdx in used_t:
                continue
            unique.append(m)
            used_q.add(m.queryIdx)
            used_t.add(m.trainIdx)
        return unique

    def run_sift_pipeline(g1, g2, mask1, mask2, nfeatures=4000, contrast=0.01, edge=20, ratio=0.8, ransac_thr=2.0, matcher_type='bf'):
        sift = cv2.SIFT_create(
            nfeatures=nfeatures,
            contrastThreshold=contrast,
            edgeThreshold=edge
        )

        kp1, des1 = sift.detectAndCompute(g1, mask1)
        kp2, des2 = sift.detectAndCompute(g2, mask2)
        if des1 is None or des2 is None:
            return None

        raw = knn_matches(des1, des2, matcher_type=matcher_type)

        good = []
        for pair in raw:
            if len(pair) < 2:
                continue
            m, n = pair
            if m.distance < ratio * n.distance:
                good.append(m)

        good = unique_matches(good)

        if len(good) < 8:
            return None

        pts1 = np.float32([kp1[m.queryIdx].pt for m in good])
        pts2 = np.float32([kp2[m.trainIdx].pt for m in good])

        method = cv2.USAC_MAGSAC if hasattr(cv2, 'USAC_MAGSAC') else cv2.FM_RANSAC
        F, maskF = cv2.findFundamentalMat(pts1, pts2, method, ransac_thr, 0.999)
        if F is None or maskF is None:
            return None

        inliers = maskF.ravel().astype(bool)
        pts1_in = pts1[inliers]
        pts2_in = pts2[inliers]
        if pts1_in.shape[0] < 8:
            return None

        errs = sampson_errors(F, pts1_in.astype(np.float64), pts2_in.astype(np.float64))
        med_err = float(np.median(errs))
        mean_err = float(np.mean(errs))

        score = pts1_in.shape[0] / (1.0 + med_err)

        return {
            'kp1': kp1,
            'kp2': kp2,
            'good': good,
            'inliers_mask': inliers,
            'pts1_in': pts1_in,
            'pts2_in': pts2_in,
            'F': F,
            'num_kp1': len(kp1),
            'num_kp2': len(kp2),
            'num_good': len(good),
            'num_inliers': int(np.sum(inliers)),
            'median_sampson': med_err,
            'mean_sampson': mean_err,
            'score': score,
            'params': (nfeatures, contrast, edge, ratio, ransac_thr, matcher_type)
        }

    best = None
    grid = {
        'nfeatures': [4000, 8000],
        'contrast': [0.02, 0.008],
        'edge': [20, 35],
        'ratio': [0.82, 0.88, 0.92],
        'ransac_thr': [2.0, 3.0, 4.0],
        'matcher_type': ['bf', 'flann']
    }

    for nf in grid['nfeatures']:
        for ct in grid['contrast']:
            for ed in grid['edge']:
                for rr in grid['ratio']:
                    for rt in grid['ransac_thr']:
                        for mt in grid['matcher_type']:
                            out = run_sift_pipeline(g1, g2, mask1, mask2, nf, ct, ed, rr, rt, mt)
                            if out is None:
                                continue
                            if best is None or out['score'] > best['score']:
                                best = out

    if best is None:
        raise RuntimeError('Nessuna configurazione valida trovata con SIFT + RANSAC')

    points1 = best['pts1_in']
    points2 = best['pts2_in']
    F = best['F']

    print('Best params (nfeatures, contrast, edge, ratio, ransac_thr, matcher):', best['params'])
    print(f"Keypoints: {best['num_kp1']} / {best['num_kp2']}")
    print(f"Good matches (unique): {best['num_good']}")
    print(f"Inliers finali: {best['num_inliers']}")
    print(f"Sampson median: {best['median_sampson']:.4f}")
    print(f"Sampson mean:   {best['mean_sampson']:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    axes[0].imshow(img1_rgb)
    axes[0].scatter(points1[:, 0], points1[:, 1], s=8, c='lime')
    axes[0].set_title(f'Image 1 - Inlier points ({len(points1)})')
    axes[0].axis('off')

    axes[1].imshow(img2_rgb)
    axes[1].scatter(points2[:, 0], points2[:, 1], s=8, c='cyan')
    axes[1].set_title(f'Image 2 - Inlier points ({len(points2)})')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    np.savez(SAVED_PATH, points1=points1, points2=points2, F=F)

# Esponi variabili compatibili con le celle successive del notebook
sift_floor_pts1 = points1.astype(np.float32)
sift_floor_pts2 = points2.astype(np.float32)
F_sift = F

### Step 2b: Manual correspondences (for algorithm validation)

In [ ]:
import numpy as np

# FOTO STORTA -> pts2
immagine_storta = np.array([
    # ===== Foto STORTA (1) =====   
    # S
    (855, 550), (752, 513), (655, 581), (780, 678), (907, 779), (809, 842), (692, 797), (689,526), (804,529), (715,662), (852,703), (749,833), (872,836),
    # D
    (1117, 532), (1273, 562), (1346, 710), (1143, 843), (1119, 697), (1190,537), (1312,604), (1335,780), (1205,851), (1123,765),(1266,841),
    # A
    (1298, 857), (1368, 687), (1438, 520), (1553, 749), (1600, 842), (1444, 757), (1327,787), (1408,593), (1506,652), (1494,756), (1579,797),
    (1473,575),(1530,699),

    # ===== Foto STORTA (2) =====
    # B
    (1550, 495), (1814, 530), (1961, 606), (1815, 683), (1540, 683), (1991, 772),
    (1818, 856), (1553, 846),(1488,500), (1604,491), (1616,603), (1624,681), (1631,773),
    (1633,848), (1546,769), (1531,597), (1904,679), (1893,604), (1916,772),(1923,548),
    (1946,657), (1956,710), (1928,848), (1880,559), (1870,669),(1900,720),(1876,847),(1859,526),(1979,816),
    # O
    (1975, 600), (2093, 713), (2070, 609), (1953, 750), (2092, 852),(1987,649), (2074,640),
    (2006,844), (2057,796), (2015,701), (2026,813), (2021,664), (2020,776),(2015,756),(2027,721),(2032,612),(2040,601),(1984,825),(2076,769),(2088,681),
    (1978,854),(2013,839),(2055,841),
    # C (prima)
    (2070,741), (2124,850), (2236,855), (2129,735), (2166,839), (2216,852), 
    (2249,807), (2163,733), (2227,740), (2195,650), (2370,682), 
    (2358,655),(2332,652), (2288,681), (2313,727), (2363,712), (2231,695), (2258,635), (2259,769),(2299,615),(2085,807),
    (2165,706),(2230,624),(2343,634),(2201,870),(2142,784),(2272,612),(2158,870),
    # C (seconda)
    (2545, 770), (2420, 868), (2358, 734), (2523, 744), (2576, 625), (2664, 697), (2420,732), (2423,807), (2469,846), (2518,814), (2375,833), (2519,848), 
    (2466,744), (2501,667), (2550,674), (2593,647), (2650,664), (2605,674), (2603,717), (2630,730),(2472,866),(2523,647),(2642,651),(2358,785),
    # O (finale)
    (2740,592),(2615, 854), (2709, 865), (2796, 869), (2855, 745), (2811, 605), (2568,802), (2563,742), (2633,788), 
    (2663,853), (2878,840), (2928,729), (2896,643), (2855,658), (2838,829),(2867,614),(2680,625),
    (2645,706),(2676,868),(2900,809),(2605,651),(2656,609),(2690,592),(2711,605),(2730,583),
    # N
    (2926, 618), (3142, 623), (3147, 850), (2942, 845), (3001,606), (2990,845), (3018,845), (2880,845), (2889,606), (2924,683), 
    (3001,684), (2921,799), (2992,802), (3070,690), (3101,617), (3178,608),
    (3101,633), (3205,634), (3206,720), (3206,850), (3256,850), (3119,850),(3150,697),(3148,807),
    (2925,605),(3213,793),(3076,650),(3141,598),
    # I
    (3320, 547), (3193, 575), (3225, 837), (3133,582), (3245,574), (3185,837), (3259,837), (3301,837), (3173,669), 
    (3252,668), (3181,778), (3255,806), (3286,530), (3283,501), (3299,470), 
    (3325,462), (3356,479), (3364,508), (3347,534),(3149,837),(3248,737),
    
    # (objects: albero, copertura, panca, fiore, paletto)
    (1869, 320), (3573, 446), (726, 876), (2028, 1770), (3209, 1567),
], dtype=np.float32)

# FOTO DRITTA -> pts1
immagine_dritta = np.array([
    # ===== Foto DRITTA (1) =====
    # S
    (735, 352), (612, 308), (492, 389), (640, 500), (786, 610), (666, 687), (522, 625), (531,320), (673,323), (551,475), (742,537), (593,671), (749,677),
    # D
    (934, 333), (1107, 359), (1208, 536), (945, 681), (927, 511), (1016,339), (1177,427), (1183,621), (1026,681), (930,595),(1107,675),
    # A
    (1193, 701), (1264, 516), (1344, 324), (1473, 584), (1531, 696),(1359, 587), (1218,634), (1303,418), (1417,465), (1409,588), (1504,638),
    (1382,395),(1452,534),

    # ===== Foto DRITTA (2) =====
    # B
    (1574, 305), (1661, 340), (1842, 427), (1678, 508), (1554, 509), (1876, 610), 
    (1689, 693), (1562, 697),(1505,310), (1645,307), (1647,421), (1652,506), (1655,612),
    (1657,699), (1560,610), (1546,424), (1781,505), (1765,428), (1796,607),(1791,360),
    (1815,485), (1839,538), (1814,688), (1743,375), (1742,493),(1779,549), (1758,682),(1723,339),(1861,658),
    # O
    (1976, 431), (2116, 554), (1996, 432), (1872, 580), (2010, 697),(1902,489), (2084,463), 
    (1928,687), (2078,644), (1943,514), (1946,643), (2030,491), (2043,613),(1941,594),(2039,558),(1945,446),(2045,427),(1899,659),(2098,615),(2110,513),
    (1994,704),(2030,687),(1985,689),
    # C (prima)
    (2131,582), (2175,699), (2299,709), (2198,571), (2215,674), (2270,698), 
    (2316,666), (2146,577), (2210,577), (2176,482), (2362,506), 
    (2348,489),(2324,490), (2280,512), (2305,568), (2356,553), (2212,531), (2246,476), (2334,610),(2280,447),(2141,654),
    (2142,541),(2210,464),(2320,460),(2240,726),(2204,636),(2258,448),(2204,720),
    # C (seconda)
    (2577, 610), (2449, 716), (2378, 570), (2450, 582), (2489, 462), (2588, 544), (2442,570), (2446,647), (2500,694), (2553,665), (2395,677), (2550,698), 
    (2386,590), (2416,506), (2467,514), (2511,484), (2572,496), (2519,514), (2517,570), (2558,579),(2506,720),(2443,482),(2559,475),(2381,632),
    # O (finale)
    (2800,437),(2671, 707), (2779, 715), (2754, 715), (2830, 587), (2799, 451), (2628,648), (2627,581), (2703,615), 
    (2725,702), (2840,690), (2905,593), (2889,500), (2839,501), (2801,672),(2850,458),(2732,475),
    (2712,546),(2731,724),(2877,661),(2660,502),(2699,464),(2740,439),(2772,449),(2787,427),
    # N
    (2941, 464), (3060, 466), (3064, 699), (2951, 700), (3015,454),(3005,700), (3025,700), (2892,700), (2900,453), (2940,526), 
    (3013,531), (2928,643), (3004,645), (2992,542), (3023,463), (3102,454),
    (3029,484), (3135,489), (3140,577), (3127,699), (3164,699), (3040,699),(3071,539),(3065,640),
    (2938,453),(3131,646),(3001,511),(3069,444),
    # I
    (3306, 401), (3322, 433), (3324, 700), (3263,434), (3379,425), (3283,700), (3362,700), (3409,700), (3300,508), 
    (3374,505), (3290,614), (3362,650), (3277,383), (3271,354), (3292,320), 
    (3320,318), (3349,330), (3354,359), (3339,390),(3242,700),(3365,590),

    # ===== Foto DRITTA (3) =====

    # (objects: albero, copertura, panca, fiore, paletto)
    (1429, 108), (3231, 290), (180, 690), (2367, 1703), (3267, 1409),
], dtype=np.float32)

# Verify counts match
assert(immagine_storta.shape == immagine_dritta.shape), "Points array shapes must match!"

# Assign to standard variable names used in subsequent steps
pts2 = immagine_storta
pts1 = immagine_dritta

print(f"Loaded {len(pts1)} manual correspondences.")

In [ ]:
# ---------------------------------------------------------------------------
# Merge: manual sign points  +  automatic SIFT points  →  unified set
# ---------------------------------------------------------------------------
# Pipeline:  raw SIFT → Sampson quality filter → spatial thinning → dedup vs manual → merge
# ---------------------------------------------------------------------------
from scipy.spatial.distance import cdist

# ── Tunable quality thresholds ──
SAMPSON_MAX    = 0.5    # keep only geometrically consistent inliers
MIN_NN_DIST    = 12.0   # spatial thinning: suppress points closer than this (px)
DEDUP_VS_MANUAL = 18.0  # remove SIFT pts too close to manual annotations

def sampson_errors_vec(F, p1, p2):
    """Vectorised Sampson error for Nx2 point arrays."""
    h1 = np.hstack([p1, np.ones((len(p1), 1))]).astype(np.float64)
    h2 = np.hstack([p2, np.ones((len(p2), 1))]).astype(np.float64)
    Fh1  = (F @ h1.T).T
    Fth2 = (F.T @ h2.T).T
    num  = np.sum(h2 * Fh1, axis=1) ** 2
    den  = Fh1[:, 0]**2 + Fh1[:, 1]**2 + Fth2[:, 0]**2 + Fth2[:, 1]**2
    return num / np.maximum(den, 1e-12)

def spatial_thin_paired(p1, p2, min_dist):
    """Greedy spatial thinning: for each point (sorted by Sampson quality),
       suppress later points within *min_dist* px."""
    keep = np.ones(len(p1), dtype=bool)
    for i in range(len(p1)):
        if not keep[i]:
            continue
        dists = np.linalg.norm(p1[i] - p1[i+1:], axis=1)
        keep[i+1:][dists < min_dist] = False
    return p1[keep], p2[keep]

# ── Step 1: Sampson quality filter ──
samp = sampson_errors_vec(F_sift, sift_floor_pts1, sift_floor_pts2)
quality_mask = samp < SAMPSON_MAX
q_pts1 = sift_floor_pts1[quality_mask].copy()
q_pts2 = sift_floor_pts2[quality_mask].copy()

# sort by Sampson so spatial thinning keeps the best ones
order = np.argsort(samp[quality_mask])
q_pts1 = q_pts1[order]
q_pts2 = q_pts2[order]

# ── Step 2: Spatial thinning ──
t_pts1, t_pts2 = spatial_thin_paired(q_pts1, q_pts2, MIN_NN_DIST)

# ── Step 3: Dedup vs manual ──
manual_pts1 = immagine_dritta.astype(np.float32)
manual_pts2 = immagine_storta.astype(np.float32)

if len(t_pts1) > 0 and len(manual_pts1) > 0:
    far_mask = np.array([np.min(np.linalg.norm(manual_pts1 - p, axis=1)) >= DEDUP_VS_MANUAL
                         for p in t_pts1])
    auto1, auto2 = t_pts1[far_mask], t_pts2[far_mask]
else:
    auto1, auto2 = t_pts1, t_pts2

# ── Step 4: Merge ──
pts1 = np.vstack([manual_pts1, auto1]).astype(np.float32)
pts2 = np.vstack([manual_pts2, auto2]).astype(np.float32)

print(f"=== Merge pipeline (SAMPSON_MAX={SAMPSON_MAX}, MIN_NN_DIST={MIN_NN_DIST}) ===")
print(f"  SIFT raw inliers:      {len(sift_floor_pts1)}")
print(f"  After Sampson < {SAMPSON_MAX}:    {len(q_pts1)}  ({100*len(q_pts1)/len(sift_floor_pts1):.0f}%)")
print(f"  After spatial thinning: {len(t_pts1)}")
print(f"  After dedup vs manual:  {len(auto1)}")
print(f"  Manual points:          {len(manual_pts1)}")
print(f"  ── Total unified:       {len(pts1)}")

In [ ]:
# ---------------------------------------------------------------------------
# Unified visualization: manual (sign) vs SIFT (automatic) correspondences
# ---------------------------------------------------------------------------
n_manual = len(manual_pts1)
n_auto   = len(pts1) - n_manual

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, img, p, title_img in zip(
        axes, [img1_rgb, img2_rgb], [pts1, pts2],
        ['Image 1 (dritta)', 'Image 2 (storta)']):
    ax.imshow(img)
    # SIFT points (plotted first so manual dots sit on top)
    ax.scatter(p[n_manual:, 0], p[n_manual:, 1],
               s=12, c='cyan', edgecolors='blue', linewidths=0.3,
               label=f'SIFT automatic ({n_auto})', alpha=0.7, zorder=2)
    # Manual points
    ax.scatter(p[:n_manual, 0], p[:n_manual, 1],
               s=13, c='red', edgecolors='darkred', linewidths=0.6,
               label=f'Manual sign ({n_manual})', alpha=0.9, zorder=3)
    ax.set_title(f'{title_img} — {len(p)} total points', fontsize=13)
    ax.legend(loc='lower right', fontsize=10)
    ax.axis('off')

fig.suptitle(f'Unified Correspondences: {n_manual} manual + {n_auto} SIFT = {len(pts1)} total',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Diagnostic: SIFT point quality analysis ──────────────────────────────────
# Evaluate whether 442 SIFT inliers are too many / too clustered / noisy

from scipy.spatial.distance import cdist

# --- 1. Sampson error per-point (using F from SIFT) ---
pts1_h = np.hstack([sift_floor_pts1, np.ones((len(sift_floor_pts1), 1))]).astype(np.float64)
pts2_h = np.hstack([sift_floor_pts2, np.ones((len(sift_floor_pts2), 1))]).astype(np.float64)
Fx1 = (F_sift @ pts1_h.T).T
Ftx2 = (F_sift.T @ pts2_h.T).T
x2Fx1 = np.sum(pts2_h * Fx1, axis=1)
denom = Fx1[:, 0]**2 + Fx1[:, 1]**2 + Ftx2[:, 0]**2 + Ftx2[:, 1]**2
sampson = (x2Fx1**2) / np.maximum(denom, 1e-12)

print("=== SIFT Inlier Quality Diagnostics ===\n")
print(f"Total SIFT inliers:    {len(sift_floor_pts1)}")
print(f"Sampson error  median: {np.median(sampson):.6f}")
print(f"Sampson error  mean:   {np.mean(sampson):.6f}")
print(f"Sampson error  90th %: {np.percentile(sampson, 90):.6f}")
print(f"Sampson error  95th %: {np.percentile(sampson, 95):.6f}")
print(f"Sampson error  max:    {np.max(sampson):.6f}")

thresholds = [0.1, 0.5, 1.0, 2.0, 5.0]
print(f"\nPoints surviving Sampson thresholds:")
for thr in thresholds:
    n_keep = int(np.sum(sampson < thr))
    print(f"  < {thr:5.1f} -> {n_keep:4d} points  ({100*n_keep/len(sampson):.1f}%)")

# --- 2. Nearest-neighbour distances (spatial clustering) ---
D = cdist(sift_floor_pts1, sift_floor_pts1)
np.fill_diagonal(D, np.inf)
nn_dist = D.min(axis=1)

print(f"\nNearest-neighbour distances (img1):")
print(f"  min :  {nn_dist.min():.1f} px")
print(f"  mean:  {nn_dist.mean():.1f} px")
print(f"  median:{np.median(nn_dist):.1f} px")
print(f"  <5 px: {np.sum(nn_dist < 5)} points (nearly duplicates)")
print(f"  <10px: {np.sum(nn_dist < 10)} points")
print(f"  <20px: {np.sum(nn_dist < 20)} points")

# --- 3. Spatial spread ---
x_range = sift_floor_pts1[:, 0].max() - sift_floor_pts1[:, 0].min()
y_range = sift_floor_pts1[:, 1].max() - sift_floor_pts1[:, 1].min()
print(f"\nSpatial spread (img1):")
print(f"  X range: {x_range:.0f} px,  Y range: {y_range:.0f} px")

# --- 4. Visualize distributions ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sampson error histogram
axes[0].hist(sampson, bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Sampson error')
axes[0].set_ylabel('Count')
axes[0].set_title('Sampson error distribution (SIFT inliers)')
axes[0].axvline(np.median(sampson), color='red', ls='--', label=f'median={np.median(sampson):.3f}')
axes[0].legend()

# NN distance histogram
axes[1].hist(nn_dist, bins=50, color='coral', edgecolor='white')
axes[1].set_xlabel('Nearest-neighbour distance (px)')
axes[1].set_ylabel('Count')
axes[1].set_title('Spatial clustering: NN distances')
axes[1].axvline(np.median(nn_dist), color='red', ls='--', label=f'median={np.median(nn_dist):.1f}')
axes[1].legend()

# Scatter: pts colored by Sampson error
sc = axes[2].scatter(sift_floor_pts1[:, 0], sift_floor_pts1[:, 1],
                     c=sampson, cmap='hot_r', s=8, vmin=0, vmax=np.percentile(sampson, 95))
axes[2].set_title('SIFT points colored by Sampson error')
axes[2].set_xlim(0, img1_rgb.shape[1])
axes[2].set_ylim(img1_rgb.shape[0], 0)
axes[2].set_aspect('equal')
plt.colorbar(sc, ax=axes[2], label='Sampson error')

plt.tight_layout()
plt.show()

### Step 3: Visualize manual points with colors

Not very useful as we added more and more points, but at least it helped us visualize how to customize better the 3d reconstruction.

In [ ]:
### NEW CELL: VISUALIZE MANUAL POINTS WITH COLORS ###

def show_manual_points(img1, img2, p1, p2):
    """Plots points side-by-side with colors and numbering."""
    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    
    axes[0].imshow(img1)
    axes[0].set_title('Image 1 (Straight image) - Annotated Points')
    axes[0].axis('off')
    
    axes[1].imshow(img2)
    axes[1].set_title('Image 2 (Rotated image) - Annotated Points')
    axes[1].axis('off')
    
    n = len(p1)
    # Generate a set of distinctive colors based on the number of points
    colors = cm.rainbow(np.linspace(0, 1, n))
    
    for i in range(n):
        # Image 1 points
        x1, y1 = p1[i]
        axes[0].plot(x1, y1, marker='o', color=colors[i], markersize=3)
        axes[0].text(x1 + 5, y1 - 10, str(i), color=colors[i], fontsize=7, weight='bold')
        
        # Image 2 points
        x2, y2 = p2[i]
        axes[1].plot(x2, y2, marker='o', color=colors[i], markersize=3)
        axes[1].text(x2 + 5, y2 - 10, str(i), color=colors[i], fontsize=7, weight='bold')

    plt.tight_layout()
    plt.show()

# Show the visualization
show_manual_points(img1_rgb, img2_rgb, pts1, pts2)

### Step 4: From fundamental matrix to essential matrix

In [ ]:
### 8-POINT ALGORITHM FOR F AND E ESTIMATION

def convert_to_homo(pts):
    ''' Converts 2D points to homogeneous coordinates (x, y, 1) '''
    return np.array([(*point, 1.0) for point in pts])

def norm_matrix(pts):
    ''' Computes the normalization matrix T for the 8-point algorithm '''
    pts = np.float64(pts)
    # Compute centroid (mean) of the points
    mean = np.mean(pts, axis=0)
    
    # Compute the average distance from the centroid
    # We want the average distance to be sqrt(2)
    scale = np.mean(la.norm(pts - mean, axis=1)) / np.sqrt(2.0)

    # Transformation matrix to shift origin to mean and scale appropriately
    T = np.array([
        [1.0 / scale, 0.0, -mean[0] / scale],
        [0.0, 1.0 / scale, -mean[1] / scale],
        [0.0, 0.0, 1.0]
    ], dtype=np.float64)

    return T

def eight_point_algorithm(pts1, pts2):
    ''' Estimates the Fundamental Matrix using the Normalized 8-Point Algorithm '''
    n = len(pts1)
    
    # STEP 1: Normalize points
    T1 = norm_matrix(pts1)
    T2 = norm_matrix(pts2)
    
    pts1_homo = convert_to_homo(pts1)
    pts2_homo = convert_to_homo(pts2)
    
    # Apply normalization (T * p)
    pts1_norm = (T1 @ pts1_homo.T).T
    pts2_norm = (T2 @ pts2_homo.T).T
    
    # STEP 2: Build the linear system matrix A
    A = np.zeros((n, 9))
    for i in range(n):
        u1, v1, _ = pts1_norm[i]
        u2, v2, _ = pts2_norm[i]
        A[i] = [u2*u1, u2*v1, u2,
                v2*u1, v2*v1, v2,
                u1,    v1,    1]
        
    # STEP 3: Solve Af = 0 using SVD
    U, S, Vt = la.svd(A)
    # The solution is the column of V (or row of V transposed) corresponding to the smallest singular value
    F_norm = Vt[-1, :].reshape(3, 3)
    
    # STEP 4: Enforce the Rank-2 constraint on F
    # Set smallest singular value to 0 to ensure rank(F) = 2
    U_f, S_f, Vt_f = la.svd(F_norm)
    S_f_corrected = S_f.copy()
    S_f_corrected[-1] = 0  # Force rank-2
    F_rank2 = U_f @ np.diag(S_f_corrected) @ Vt_f
    
    # STEP 5: Denormalize to get the final F
    F = T2.T @ F_rank2 @ T1
    
    # Standardize by dividing by the bottom-right element
    F = F / F[2, 2]
    
    return F

# Compute Matrices

# 1. Compute Fundamental Matrix using our custom function
F = eight_point_algorithm(pts1, pts2)

print("Fundamental Matrix F (Custom 8-Point):")
print(F)

# 2. Compute Essential Matrix E from F and the intrinsic matrix K
E = K.T @ F @ K

print("\nEssential Matrix E:")
print(E)

# 3. Recover relative Rotation (R) and Translation (t) using OpenCV 
# (Pose recovery from E involves complex Chirality checks, so keeping cv2 here is standard)
_, R, t, _ = cv2.recoverPose(E, pts1, pts2, K)

print("\nRotation Matrix R:")
print(R)

print("\nTranslation Vector t:")
print(t)

### Comparison with OpenCV Five-Point Algorithm

We need to compare our 8-point algorithm with OpenCV's 5-point algorithm.
The five-point algorithm directly computes the Essential Matrix E for calibrated cameras.

In [ ]:
# Compute Essential Matrix using OpenCV's five-point algorithm with RANSAC
E_opencv, mask_opencv = cv2.findEssentialMat(pts1, pts2, K, method=cv2.RANSAC, 
                                              prob=0.999, threshold=1.0)

print("\n=== COMPARISON: Custom 8-Point vs OpenCV 5-Point ===")
print("\nEssential Matrix E (Custom 8-Point):")
print(E)
print("\nEssential Matrix E (OpenCV 5-Point with RANSAC):")
print(E_opencv)

# Recover pose from OpenCV's Essential Matrix
_, R_opencv, t_opencv, mask_pose_opencv = cv2.recoverPose(E_opencv, pts1, pts2, K)

print("\n--- Rotation Matrix Comparison ---")
print("R (Custom):")
print(R)
print("\nR (OpenCV):")
print(R_opencv)
print(f"\nRotation difference (Frobenius norm): {la.norm(R - R_opencv):.6f}")

print("\n--- Translation Vector Comparison ---")
print("t (Custom):")
print(t.ravel())
print("\nt (OpenCV):")
print(t_opencv.ravel())
print(f"\nTranslation angular difference: {np.arccos(np.clip(np.dot(t.ravel(), t_opencv.ravel()), -1, 1)) * 180 / np.pi:.4f} degrees")

# Count inliers from RANSAC
inliers_count = np.sum(mask_opencv)
print(f"\n--- RANSAC Inliers ---")
print(f"Inliers: {inliers_count}/{len(pts1)} points")
print(f"Outliers: {len(pts1) - inliers_count} points")

if inliers_count < len(pts1):
    print("\nNote: RANSAC detected outliers. With manual annotation, this suggests")
    print("measurement errors or poor feature localization in some correspondences.")

### Discussion: 8-Point vs 5-Point Algorithm

Just a summary to keep track of what we are doing, the discussion is fully explained in the report.

**Key Differences:**

1. **Minimal Set**: 
   - 5-point algorithm: Requires only 5 correspondences (minimal)
   - 8-point algorithm: Requires at least 8 correspondences

2. **Direct Computation**:
   - 5-point: Directly computes Essential matrix E (for calibrated cameras)
   - 8-point: Computes Fundamental matrix F, then E = K^T F K

3. **Robustness**:
   - 5-point with RANSAC: Automatically handles outliers
   - 8-point: Assumes all correspondences are correct (least squares)

4. **Accuracy**:
   - With perfect correspondences: Similar results
   - With noise/outliers: 5-point + RANSAC is more robust
   - With many good points: 8-point can be more stable

**Expected Results:**
- If all manual correspondences are accurate: Results should be very similar
- If some correspondences have errors: OpenCV may flag them as outliers
- The Essential Matrix should satisfy the same epipolar constraint in both cases

### Step 5: Computer epipolar error

In [ ]:
def convert_to_homo(pts):
    ''' Converts 2D points to homogeneous coordinates. '''
    return [(*point, 1) for point in pts]

def compute_epipolar_error(pts1, pts2, F):
    pts1_homo = np.array(convert_to_homo(pts1))
    pts2_homo = np.array(convert_to_homo(pts2))
    n = len(pts1)
    errors = np.zeros(n)
    
    for i in range(n):
        line2 = F @ pts1_homo[i]
        line1 = F.T @ pts2_homo[i]
        
        num1 = abs(pts1_homo[i].T @ F.T @ pts2_homo[i])
        num2 = abs(pts2_homo[i].T @ F @ pts1_homo[i])
        
        denom1 = np.sqrt(line1[0]**2 + line1[1]**2)
        denom2 = np.sqrt(line2[0]**2 + line2[1]**2)
        
        d1 = num1 / denom1
        d2 = num2 / denom2
        errors[i] = (d1 + d2) / 2  # average of distances

    return errors.mean(), errors

# Pass the manual points to calculate the error
mean_error, errors = compute_epipolar_error(pts1, pts2, F)

print(f"\nMean epipolar distance: {mean_error:.6f} pixels")
print(f"Max epipolar distance: {errors.max():.6f} pixels")
print(f"Min epipolar distance: {errors.min():.6f} pixels")

### Step 6: Plot epipolar lines

In [ ]:
def plot_epipolar_lines(img1, img2, pts1, pts2, F):
    fig, axes = plt.subplots(1, 2, figsize=(15, 8))
    n = len(pts1)
    axes[0].imshow(img1)
    axes[0].set_title('Image 1')
    axes[0].axis('off')
    axes[1].imshow(img2)
    axes[1].set_title('Image 2')
    axes[1].axis('off')

    pts1_homo = np.array(convert_to_homo(pts1))
    pts2_homo = np.array(convert_to_homo(pts2))

    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    def plot_line_safely(ax, w, h, line, color='r'):
        a, b, c = line
        if abs(b) < 1e-10:
            return False

        x_intersects, y_intersects = [], []

        if b != 0:
            x_top = -c / a if a != 0 else float('inf')
            if 0 <= x_top <= w:
                x_intersects.append(x_top); y_intersects.append(0)

        x_bottom = -(b * h + c) / a if a != 0 else float('inf')
        if 0 <= x_bottom <= w:
            x_intersects.append(x_bottom); y_intersects.append(h)

        y_left = -c / b
        if 0 <= y_left <= h:
            x_intersects.append(0); y_intersects.append(y_left)

        y_right = -(a * w + c) / b
        if 0 <= y_right <= h:
            x_intersects.append(w); y_intersects.append(y_right)

        if len(x_intersects) >= 2:
            ax.plot(x_intersects, y_intersects, color, linewidth=0.15, alpha=0.5)
            return True
        return False

    # Plot lines for IMAGE 2
    for i in range(n):
        line = F @ pts1_homo[i]
        plot_line_safely(axes[1], w2, h2, line, 'r')

    # Plot lines for IMAGE 1
    for i in range(n):
        line = F.T @ pts2_homo[i]
        plot_line_safely(axes[0], w1, h1, line, 'b')

    # Plot points
    axes[0].plot(np.array(pts1)[:, 0], np.array(pts1)[:, 1], 'go', markersize=2)
    axes[1].plot(np.array(pts2)[:, 0], np.array(pts2)[:, 1], 'go', markersize=2)

    plt.tight_layout()
    plt.show()

# We pass all manual points directly to the plotting function
plot_epipolar_lines(img1_rgb, img2_rgb, pts1, pts2, F)

## Part 3: 3D Reconstruction and Visualization (2 pts)

In this section we:
1. Use the recovered camera poses (R, t) to triangulate the corresponding points
2. Reconstruct the 3D structure of the scene
3. Visualize the reconstructed 3D points from multiple viewpoints

The camera matrices are:
- **Camera 1 (reference)**: P1 = K [I | 0] (identity pose)
- **Camera 2 (relative)**: P2 = K [R | t] (recovered from Essential matrix)

In [ ]:
# ── Use OpenCV 5-point pose for triangulation ──────────────────────────

R, t = R_opencv, t_opencv

# Build projection matrices for both cameras
# Camera 1 is at the origin (world reference frame)
P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])

# Camera 2 has the relative pose [R | t]
P2 = K @ np.hstack([R, t])

print("Projection Matrix P1 (Camera 1 - Reference):")
print(P1)
print("\nProjection Matrix P2 (Camera 2 - Relative pose):")
print(P2)

# Triangulate the corresponding points
# cv2.triangulatePoints expects 2xN arrays of image points
pts1_for_triangulation = pts1.T  # Shape: (2, N)
pts2_for_triangulation = pts2.T  # Shape: (2, N)

# Triangulate: returns 4xN homogeneous coordinates
points_4d_homogeneous = cv2.triangulatePoints(P1, P2, pts1_for_triangulation, pts2_for_triangulation)

# Convert from homogeneous to 3D Cartesian coordinates
points_3d = points_4d_homogeneous[:3, :] / points_4d_homogeneous[3, :]
points_3d = points_3d.T  # Shape: (N, 3)

print(f"\n3D Points Reconstructed: {points_3d.shape[0]} points")
print("\nFirst 5 reconstructed 3D points:")
print(points_3d[:5])

# Check reconstruction quality: reproject points back to images
def reproject_points(points_3d, P):
    """Reproject 3D points to image using projection matrix P"""
    points_3d_homo = np.hstack([points_3d, np.ones((points_3d.shape[0], 1))])
    projected_homo = (P @ points_3d_homo.T).T
    projected_2d = projected_homo[:, :2] / projected_homo[:, 2:3]
    return projected_2d

reprojected_pts1 = reproject_points(points_3d, P1)
reprojected_pts2 = reproject_points(points_3d, P2)

# Compute reprojection errors
reproj_error_1 = np.mean(np.linalg.norm(pts1 - reprojected_pts1, axis=1))
reproj_error_2 = np.mean(np.linalg.norm(pts2 - reprojected_pts2, axis=1))

print(f"\nReprojection Error Camera 1: {reproj_error_1:.4f} pixels")
print(f"Reprojection Error Camera 2: {reproj_error_2:.4f} pixels")
print(f"Average Reprojection Error: {(reproj_error_1 + reproj_error_2) / 2:.4f} pixels")

In [ ]:
# 3D Visualization of the reconstructed scene
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(16, 12))

# We'll create 4 different viewpoints of the 3D reconstruction
viewpoints = [
    (30, 45, "View 1: Elevation 30°, Azimuth 45°"),
    (20, 135, "View 2: Elevation 20°, Azimuth 135°"),
    (60, -45, "View 3: Elevation 60°, Azimuth -45°"),
    (10, 90, "View 4: Elevation 10°, Azimuth 90°")
]

for idx, (elev, azim, title) in enumerate(viewpoints, 1):
    ax = fig.add_subplot(2, 2, idx, projection='3d')
    
    # Plot the reconstructed 3D points
    ax.scatter(points_3d[:, 0], points_3d[:, 1], points_3d[:, 2], 
               c='red', marker='o', s=5, label='3D Points', alpha=0.6)
    
    # Plot camera centers
    # Camera 1 is at origin
    cam1_center = np.array([0, 0, 0])
    ax.scatter(*cam1_center, c='blue', marker='^', s=120, label='Camera 1')
    
    # Camera 2 center: C2 = -R^T * t
    cam2_center = -R.T @ t
    ax.scatter(*cam2_center.ravel(), c='green', marker='^', s=200, label='Camera 2')
    
    # Draw camera optical axes
    axis_length = np.max(np.abs(points_3d)) * 0.3
    ax.scatter(*cam2_center.ravel(), c='green', marker='^', s=120, label='Camera 2')
    # Camera 1 optical axis (Z-axis in world frame)
    ax.quiver(0, 0, 0, 0, 0, axis_length, color='blue', arrow_length_ratio=0.1, linewidth=2)
    
    # Camera 2 optical axis (R.T maps camera z-axis to world frame)
    cam2_z_axis = R.T @ np.array([0, 0, axis_length])
    ax.quiver(*cam2_center.ravel(), *cam2_z_axis.ravel(), 
              color='green', arrow_length_ratio=0.1, linewidth=2)
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title)
    ax.legend()
    ax.view_init(elev=elev, azim=azim)
    
    # Set equal aspect ratio for better visualization
    max_range = np.array([points_3d[:, 0].max() - points_3d[:, 0].min(),
                          points_3d[:, 1].max() - points_3d[:, 1].min(),
                          points_3d[:, 2].max() - points_3d[:, 2].min()]).max() / 2.0
    
    mid_x = (points_3d[:, 0].max() + points_3d[:, 0].min()) * 0.5
    mid_y = (points_3d[:, 1].max() + points_3d[:, 1].min()) * 0.5
    mid_z = (points_3d[:, 2].max() + points_3d[:, 2].min()) * 0.5
    
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)

    zoom = max_range * 0.55
    ax.set_xlim(mid_x - zoom, mid_x + zoom)
    ax.set_ylim(mid_y - zoom, mid_y + zoom)

In [ ]:
# Additional visualization: Top view and side view
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top view (X-Y plane)
axes[0].scatter(points_3d[:, 0], points_3d[:, 1], c='red', s=8, alpha=0.6, edgecolors='darkred', linewidths=0.3)

# Plot cameras
axes[0].scatter(0, 0, c='blue', marker='^', s=150, label='Camera 1', edgecolors='black', linewidths=1.5)
cam2_center = -R.T @ t
axes[0].scatter(cam2_center[0], cam2_center[1], c='green', marker='^', s=150,
                label='Camera 2', edgecolors='black', linewidths=1.5)

axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].set_title('Top View (X-Y Plane)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

# Side view (X-Z plane)
axes[1].scatter(points_3d[:, 0], points_3d[:, 2], c='red', s=8, alpha=0.6, edgecolors='darkred', linewidths=0.3)

axes[1].scatter(0, 0, c='blue', marker='^', s=150, label='Camera 1', edgecolors='black', linewidths=1.5)
axes[1].scatter(cam2_center[0], cam2_center[2], c='green', marker='^', s=150,
                label='Camera 2', edgecolors='black', linewidths=1.5)

axes[1].set_xlabel('X')
axes[1].set_ylabel('Z')
axes[1].set_title('Side View (X-Z Plane)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Print statistics about the 3D reconstruction
print("=== 3D Reconstruction Statistics ===")
print(f"\nNumber of reconstructed points: {len(points_3d)}")
print(f"\nScene bounding box:")
print(f"  X: [{points_3d[:, 0].min():.2f}, {points_3d[:, 0].max():.2f}] (range: {points_3d[:, 0].max() - points_3d[:, 0].min():.2f})")
print(f"  Y: [{points_3d[:, 1].min():.2f}, {points_3d[:, 1].max():.2f}] (range: {points_3d[:, 1].max() - points_3d[:, 1].min():.2f})")
print(f"  Z: [{points_3d[:, 2].min():.2f}, {points_3d[:, 2].max():.2f}] (range: {points_3d[:, 2].max() - points_3d[:, 2].min():.2f})")

print(f"\nCamera separation (baseline): {np.linalg.norm(cam2_center):.2f} units")

# Verify points are in front of both cameras
# For camera 1: points should have positive Z
points_in_front_cam1 = np.sum(points_3d[:, 2] > 0)
print(f"\nPoints in front of Camera 1: {points_in_front_cam1}/{len(points_3d)}")

# For camera 2: transform points to camera 2 frame and check Z
points_in_cam2_frame = (R @ points_3d.T + t).T
points_in_front_cam2 = np.sum(points_in_cam2_frame[:, 2] > 0)
print(f"Points in front of Camera 2: {points_in_front_cam2}/{len(points_3d)}")

if points_in_front_cam1 == len(points_3d) and points_in_front_cam2 == len(points_3d):
    print("\n✓ All points are correctly in front of both cameras (chirality check passed)")
else:
    print("\n⚠ Warning: Some points are behind one or both cameras")

### 3D Animated Visualization (Interactive Rotation)

We need several views of the scene and a 3D gif. Here we create an interactive rotating animation which should satisfy both.

In [ ]:
# Create 3D animated rotation (as requested in instructions)
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

# Setup figure for animation
fig_anim = plt.figure(figsize=(10, 8))
ax_anim = fig_anim.add_subplot(111, projection='3d')

def init_animation():
    """Initialize animation"""
    ax_anim.clear()
    return []

def animate(frame):
    """Animation function - rotates the view"""
    ax_anim.clear()
    
    # Plot 3D points
    ax_anim.scatter(points_3d[:, 0], points_3d[:, 1], points_3d[:, 2], 
                   c='red', marker='o', s=5, label='3D Points', alpha=0.6, edgecolors='darkred', linewidths=0.3)
    
    # Plot camera 1 at origin
    ax_anim.scatter(0, 0, 0, c='blue', marker='^', s=150, label='Camera 1', edgecolors='darkblue', linewidths=1.5)
    
    # Plot camera 2
    cam2_center_anim = -R.T @ t
    ax_anim.scatter(*cam2_center_anim.ravel(), c='green', marker='^', s=300, label='Camera 2', edgecolors='darkgreen', linewidths=2)
    
    # Draw optical axes
    axis_length = np.max(np.abs(points_3d)) * 0.3
    ax_anim.scatter(*cam2_center_anim.ravel(), c='green', marker='^', s=150, label='Camera 2', edgecolors='darkgreen', linewidths=1.5)
    cam2_z_axis = R.T @ np.array([0, 0, axis_length])
    ax_anim.quiver(*cam2_center_anim.ravel(), *cam2_z_axis.ravel(), 
                   color='green', arrow_length_ratio=0.15, linewidth=2.5, alpha=0.7)
    
    # Set labels and title
    ax_anim.set_xlabel('X', fontsize=11, fontweight='bold')
    ax_anim.set_ylabel('Y', fontsize=11, fontweight='bold')
    ax_anim.set_zlabel('Z', fontsize=11, fontweight='bold')
    ax_anim.set_title(f'3D Reconstruction - Frame {frame}/120', fontsize=13, fontweight='bold')
    ax_anim.legend(loc='upper right')
    
    # Rotate view - complete 360 degree rotation
    angle = frame * 3  # 3 degrees per frame = 360 degrees in 120 frames
    ax_anim.view_init(elev=20, azim=angle)
    
    # Set equal aspect ratio
    max_range = np.array([points_3d[:, 0].max() - points_3d[:, 0].min(),
                          points_3d[:, 1].max() - points_3d[:, 1].min(),
                          points_3d[:, 2].max() - points_3d[:, 2].min()]).max() / 2.0
    
    mid_x = (points_3d[:, 0].max() + points_3d[:, 0].min()) * 0.5
    mid_y = (points_3d[:, 1].max() + points_3d[:, 1].min()) * 0.5
    mid_z = (points_3d[:, 2].max() + points_3d[:, 2].min()) * 0.5
    
    ax_anim.set_xlim(mid_x - max_range, mid_x + max_range)
    ax_anim.set_ylim(mid_y - max_range, mid_y + max_range)
    ax_anim.set_zlim(mid_z - max_range, mid_z + max_range)
    
    zoom = max_range * 0.55
    ax_anim.set_xlim(mid_x - zoom, mid_x + zoom)
    ax_anim.set_ylim(mid_y - zoom, mid_y + zoom)
    ax_anim.set_zlim(mid_z - zoom, mid_z + zoom)
    
    ax_anim.grid(True, alpha=0.3)
    
    return []

# Create animation - 120 frames for smooth 360° rotation
print("Creating 3D rotation animation...")
anim = FuncAnimation(fig_anim, animate, init_func=init_animation, frames=120, interval=50, blit=False)

# Render the first frame so the static notebook output is not empty
animate(0)
plt.show()

# Display animation inline (if supported)
try:
    # This works in Jupyter notebooks
    HTML(anim.to_jshtml())
except:
    # If HTML display fails, just show a message
    print("Animation created successfully. Use the slider below to rotate the view.")

### Optional: Save Animation as GIF

Uncomment the code below to save the animation as a GIF file.

Note: This requires the `pillow` package and may take a minute to render.

In [ ]:
# # Uncomment to save animation as GIF
# print("Saving animation as GIF...")
# writer = PillowWriter(fps=20)
# anim.save('3d_reconstruction_animation.gif', writer=writer, dpi=80)
# print("✓ Animation saved as '3d_reconstruction_animation.gif'")

# # Alternative: save as MP4 (requires ffmpeg)
# # from matplotlib.animation import FFMpegWriter
# # writer = FFMpegWriter(fps=20, bitrate=1800)
# # anim.save('3d_reconstruction.mp4', writer=writer, dpi=100)

### Reconstructing Letter Curves (Enhanced Detail)

The instructions hint at reconstructing the corners and curves of letters. With the current manual correspondences,
we can visualize the reconstructed structure. For a more detailed model, we would need:

1. **More correspondence points** along the letter edges and curves
2. **Dense feature matching** (SIFT/SURF) instead of just 12 manual points
3. **Edge detection and curve fitting** in 2D, then triangulation to 3D

Below we create an enhanced visualization showing the potential for curve reconstruction.

In [ ]:
# Enhanced 3D visualization with improved styling for letter structure
fig_enhanced = plt.figure(figsize=(16, 10))

# Create two subplots: full 3D view and close-up
ax1 = fig_enhanced.add_subplot(121, projection='3d')
ax2 = fig_enhanced.add_subplot(122, projection='3d')

for ax, view_title in zip([ax1, ax2], ['Full Scene', 'Close-up View']):
    # Plot reconstructed points with enhanced styling
    scatter = ax.scatter(points_3d[:, 0], points_3d[:, 1], points_3d[:, 2],
                        c=points_3d[:, 2], cmap='viridis', marker='o', s=8,
                        label='Reconstructed Points', alpha=0.7, edgecolors='black', linewidths=0.3)

    # Draw connections to hint at letter structure
    ax.plot(points_3d[:, 0], points_3d[:, 1], points_3d[:, 2],
           'gray', alpha=0.2, linewidth=1, linestyle=':', label='Point connections')

    # Plot cameras with enhanced visibility
    ax.scatter(0, 0, 0, c='cyan', marker='^', s=150, label='Camera 1',
              edgecolors='blue', linewidths=2, alpha=0.9)

    cam2_center_plot = -R.T @ t
    ax.scatter(*cam2_center_plot.ravel(), c='lime', marker='^', s=150,
              label='Camera 2', edgecolors='green', linewidths=2, alpha=0.9)

    # Draw viewing frustums (simplified)
    axis_length = np.max(np.abs(points_3d)) * 0.4
    ax.quiver(0, 0, 0, 0, 0, axis_length, color='cyan', arrow_length_ratio=0.1, linewidth=3, alpha=0.6)
    cam2_z_axis = R.T @ np.array([0, 0, axis_length])
    ax.quiver(*cam2_center_plot.ravel(), *cam2_z_axis.ravel(),
             color='lime', arrow_length_ratio=0.1, linewidth=3, alpha=0.6)

    # Styling
    ax.set_xlabel('X (mm)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Y (mm)', fontsize=11, fontweight='bold')
    ax.set_zlabel('Z (mm)', fontsize=11, fontweight='bold')
    ax.set_title(f'{view_title} - Letter Structure', fontsize=13, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='--')

    # Compute bounding box
    max_range = np.array([points_3d[:, 0].max() - points_3d[:, 0].min(),
                          points_3d[:, 1].max() - points_3d[:, 1].min(),
                          points_3d[:, 2].max() - points_3d[:, 2].min()]).max() / 2.0
    mid_x = (points_3d[:, 0].max() + points_3d[:, 0].min()) * 0.5
    mid_y = (points_3d[:, 1].max() + points_3d[:, 1].min()) * 0.5
    mid_z = (points_3d[:, 2].max() + points_3d[:, 2].min()) * 0.5

    # Set viewing angle
    if 'Close-up' in view_title:
        ax.view_init(elev=15, azim=60)
        margin = max_range * 0.4
        ax.set_xlim(mid_x - margin, mid_x + margin)
        ax.set_ylim(mid_y - margin, mid_y + margin)
        ax.set_zlim(mid_z - margin, mid_z + margin)
    else:
        ax.view_init(elev=25, azim=45)
        zoom = max_range * 0.55
        ax.set_xlim(mid_x - zoom, mid_x + zoom)
        ax.set_ylim(mid_y - zoom, mid_y + zoom)
        ax.set_zlim(mid_z - zoom, mid_z + zoom)

    # Add colorbar for depth
    if ax == ax1:
        cbar = plt.colorbar(scatter, ax=ax, shrink=0.6, aspect=10)
        cbar.set_label('Depth (Z)', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Enhanced 3D visualization complete")
print(f"✓ Reconstructed {len(points_3d)} correspondence points")

In [ ]:
import plotly.graph_objects as go

# Build interactive 3D scatter with Plotly
fig = go.Figure()

# Hand-picked maximally-distinct palette for each letter group
_GROUPS = [
    (13,  '#E03C31'),   # S  — vivid red
    (24,  '#2D7DD2'),   # D  — strong blue
    (37,  '#F5A623'),   # A  — warm orange
    (66,  '#44BBA4'),   # B  — teal
    (89,  '#8B5CF6'),   # O  — violet
    (117, '#E36BAE'),   # C1 — hot pink
    (141, '#17BEBB'),   # C2 — cyan
    (166, '#3D5A80'),   # O2 — slate blue
    (194, '#97CC04'),   # N  — lime green
    (215, '#F7B32B'),   # I  — golden yellow
]
_grass_hex = '#4a7c59'                            # natural muted green for objects

colors = []
for i in range(len(points_3d)):
    assigned = False
    for upper, hex_col in _GROUPS:
        if i < upper:
            colors.append(hex_col)
            assigned = True
            break
    if not assigned:
        colors.append(_grass_hex)

# Triangulated points coloured by group, thin black border for clarity
fig.add_trace(go.Scatter3d(
    x=points_3d[:, 0], y=points_3d[:, 1], z=points_3d[:, 2],
    mode='markers',
    marker=dict(
        size=3,
        color=colors,
        opacity=0.9,
        line=dict(width=0.5, color='black'),
    ),
    name='3D Points'
))

# Camera 1 at origin
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode='markers',
    marker=dict(size=2.5, color='red', symbol='diamond'),
    name='Camera 1'
))

# Camera 2
cam2_pos = -R_opencv.T @ t_opencv
fig.add_trace(go.Scatter3d(
    x=[cam2_pos[0, 0]], y=[cam2_pos[1, 0]], z=[cam2_pos[2, 0]],
    mode='markers',
    marker=dict(size=2.5, color='blue', symbol='diamond'),
    name='Camera 2'
))

fig.update_layout(
    title='Interactive 3D Reconstruction (drag to rotate, scroll to zoom)',
    scene=dict(
        xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
        aspectmode='data'
    ),
    width=900, height=700
)
fig.show()